In [1]:
import warnings
warnings.filterwarnings('ignore')

# 검색 증강 생성(Retrieval Augmented Generation, RAG)

RAG 기법은 기존의 대규모 언어 모델(LLM)을 확장하여, 주어진 컨테스트나 질문에 대해 더욱 정확하고 풍부한 정보를 제공하는 방법으로 모델이 학습 데이터에 포함되지 않은 외부 데이터를 실시간으로 검색(retrieval)하고, 이를 바탕으로 답변을 생성(generation)하는 과정을 포함한다. 특히 생성된 내용이 사실이 아닌 것으로 오인되는 현상이나 생성형 AI나 LLM이 사실이 아니거나 터무니없는 내용을 진실처럼 생성해 내는 환각(hallucination)이라는 현상을 방지하고, 모델이 최신 정보를 반영하거나 더 넓은 지식을 활용할 수 있게 한다.

## RAG의 기본 구조

`검색 단계(retrieval phase)`  
사용자의 질문이나 컨텍스트를 입력으로 받아서, 이와 관련된 외부 데이터를 검색하는 단계로 검색 엔진이나 데이터베이스 등 다양한 소스에서 필요한 정보를 찾아낸다.  
검색된 데이터는 질문에 대한 답변을 생성하는데 적합하고 상세한 정보를 포함하는 것을 목표로 한다.

`생성 단계(generation phase)`  
검색된 데이터를 기반으로 LLM 모델이 사용자의 질문에 답변을 생성하는 단계로 모델은 검색된 정보와 기존 지식을 결합하여, 주어진 질문에 대한 답변을 생성한다.

## RAG의 장점

`풍부한 정보 제공`: RAG는 검색을 통해 얻은 외부 데이터를 활용하여, 보다 구체적이고 풍부한 정보를 제공한다.  
`실시간 정보 반영`: 최신 데이터를 검색하여 반영함으로써, RAG가 실시간으로 변화하는 정보에 대응할 수 있다.  
`환각 방지`: 검색을 통해 실제 데이터에 기반한 답변을 생성함으로써, 환각 현상이 발생할 위험을 줄이 정확도를 높일 수 있다.

<img src="./rag-graphic.png" width="900" align="left" />

모델의 학습 데이터에 포함되지 않은 데이터를 사용한다.(환각 방지)  
외부 데이터를 검색한 후 생성 단계에서 LLM에 전달한다.  
모델은 주어진 컨텍스트나 질문에 더 적합하고 풍부한 정보를 반영한 답변을 생성한다.

`langchain`: 메인 프레임워크로 LLM과 프롬프트, 도구들을 연결하여 체인(chain)을 구성한다.  
`langchain-openai`: OpenAI 전용 모듈로 GPT-3.5, GPT-4o 등 OpenAI 모델을 LangChain에 쉽게 쓸 수 있게 연결해준다.  
`langchain-community`: 커뮤니티 지원 도구로 Google, HuggingFace 등 서드파티 서비스나 다양한 DB 연결 도구들이 모여있는 저장소이다.  
`langchain-text-splitters`: 텍스트 절단기로 긴 문서를 AI가 읽이 좋게 작은 조각(Chunk)으로 나눈다.  
`tiktoken`: 토큰 계산기로 OpenAI 모델이 텍스트를 처리할 때 사용하는 토큰 개수를 계산한다. 비용 예측 및 길이 제한 관리에 사용한다.  
`chromadb`: 벡터 데이터베이스(Vector DB)로 텍스트를 벡터(숫자)로 변환해서 저장하고, 질문과 유사한 정보를 찾아내는 RAG 구현의 핵심 도구이다.

In [2]:
# !pip install -q langchain langchain-openai langchain-community langchain-text-splitters tiktoken chromadb

In [3]:
import os

# os.environ['OPENAI_API_KEY'] = '사용자 API Key'


네트워크 요청 시 애플리케이션을 식별하기 위한 식별자(USER_AGENT)를 정의한다.  
웹 서버나 API는 출처가 불분명한 자동화 스크립트의 접근을 차단하는 경우가 많아서 USER_AGENT environment variable not set, consider setting it to identify your requests.와 같은 메시지가 출력되면 올바른 식별 정보를 제공하여 정상적인 요청으로 인식하게 한다.

In [4]:
os.environ['USER_AGENT'] = 'MyAppName/1.0'

# RAG 파이프라인

## 데이터 읽기(Load Data)

웹 페이지의 데이터를 읽어오기 위해 WebBaseLoader를 import 한다.

In [5]:
from langchain_community.document_loaders import WebBaseLoader

<img src="./위키백과_정책과지침.png" width="1100" align="left" />

In [6]:
# 데이터를 읽어올 웹 페이지를 지정한다.
url = 'https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EC%A0%95%EC%B1%85%EA%B3%BC_%EC%A7%80%EC%B9%A8'
# WebBaseLoader() 클래스의 인수로 데이터를 읽어올 웹 페이지의 주소를 넘겨서 웹 페이지 데이터를 읽어온다.
loader = WebBaseLoader(url)

# load() 메소드로 읽어온 웹 페이지에서 텍스트만 얻어온다.
docs = loader.load()

print(len(docs))
print(len(docs[0].page_content))
print(docs[0].page_content[5000:6000])

1
13347
도 있지요.
특정 사용자가 공동체의 규범을 총체적으로 어기고 있다면 규범 준수를 위해 좀 더 빠르게 강력한 수단을 이용해야 합니다. 특히 정책 문서에 명시된 원칙을 지키지 않는 것은 대부분의 경우 다른 사용자에게 받아들여지지 않습니다 (다른 분들에게 예외 상황임을 설득할 수 있다면 가능하기는 하지만요). 이는 당신을 포함해서 편집자 개개인이 정책과 지침을 직접 집행 및 적용한다는 것을 의미합니다.
특정 사용자가 명백히 정책에 반하는 행동을 하거나 정책과 상충되는 방식으로 지침을 어기는 경우, 특히 의도적이고 지속적으로 그런 행위를 하는 경우 해당 사용자는 관리자의 제재 조치로 일시적, 혹은 영구적으로 편집이 차단될 수 있습니다. 영어판을 비롯한 타 언어판에서는 일반적인 분쟁 해결 절차로 끝낼 수 없는 사안은 중재위원회가 개입하기도 합니다.
문서 내용
정책과 지침의 문서 내용은 처음 읽는 사용자라도 원칙과 규범을 잘 이해할 수 있도록 다음 원칙을 지켜야 합니다.
명확하게 작성하세요. 소수만 알아듣거나 준법률적인 단어, 혹은 지나치게 단순한 표현은 피해야 합니다. 명확하고, 직접적이고, 모호하지 않고, 구체적으로 작성하세요. 지나치게 상투적인 표현이나 일반론은 피하세요. 지침, 도움말 문서 및 기타 정보문 문서에서도 "해야 합니다" 혹은 "하지 말아야 합니다" 같이 직접적인 표현을 굳이 꺼릴 필요는 없습니다.
가능한 간결하게, 너무 단순하지는 않게. 정책이 중언부언하면 오해를 부릅니다. 불필요한 말은 생략하세요. 직접적이고 간결한 설명이 마구잡이식 예시 나열보다 더 이해하기 쉽습니다. 각주나 관련 문서 링크를 이용하여 더 상세히 설명할 수도 있습니다.
규칙을 만든 의도를 강조하세요. 사용자들이 상식대로 행동하리라 기대하세요. 정책의 의도가 명료하다면, 추가 설명은 필요 없죠. 즉 규칙을 '어떻게' 지키는지와 더불어 '왜' 지켜야 하는지 확실하게 밝혀야 합니다.
범위는 분명히, 중복은 피하기. 되도록 앞부분에서 정책 및 지침의 목적과 범위를 분명하게 

## 작은 청크 단위로 분할(Text Split)

긴 텍스트를 모델이 처리하기 좋은 작은 단위(chunk)로 잘라주기 위해서 RecursiveCharacterTextSplitter를 import 한다.

RAG나 LLM 파이프라인을 구축할 때 문서가 너무 길면 토큰 제한을 초과하거나 검색 정확도가 떨어진다. 문맥이 어색하게 끊어지지 않도록 단락(문단) => 문장 => 단어 => 글자 순서로 우선순위를 두어 재귀적으로 잘라준다.

단락(문단)`\n\n` => 문장`\n` => 단어`' '` => 글자`''`

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

RecursiveCharacterTextSplitter 클래스는 문서를 적절한 크기로 자르고(청크) 인접한 청크간에 문맥 정보가 손실되는 것을 방지하기 위해 겹치는 정도를 지정해서 문서를 청크로 나눈다.  
`chunk_size`: 하나의 청크가 가질 수 있는 최대 문자 수를 지정한다.  
`chunk_overlap`: 인접한 청크간에 문장이 잘리는 경계선에서 문맥 정보가 손실되는 것을 방지하기 위해서 겹치게 만드는 문자 수를 지정한다.

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# split_documents() 메소드로 웹 페이지에서 읽어온 데이터가 저장된 Document 객체를 넘겨서 청크로 나눈다.
splits = text_splitter.split_documents(docs)

print(len(splits))
print(splits[14])

18
page_content='정책 및 지침을 과감히 편집할 시 되돌리기를 당했다면 먼저 되돌리기 하는 것 보다, 토론을 열어 의견을 나누는 것이 좋습니다. 토론 중 자신의 주장을 뒷받침하기 위한 정책 편집은 체제를 시험하는 것으로 보일 수 있습니다. 특히 편집 시 논쟁에 자신이 참여한 사실을 밝히지 않았다면 더더욱이요.
격하
특정 정책이나 지침이 편집 관행이나 공동체 규범이 바뀌며 쓸모없어질 수 있고, 다른 문서가 개선되어 내용이 중복될 수 있으며, 불필요한 내용이 증식할 수도 있습니다. 이 경우 편집자들은 정책을 지침으로 격하하거나, 정책 또는 지침을 보충 설명, 정보문, 수필 또는 중단 문서로 격하할 것을 제안할 수 있습니다. 
격하 과정은 채택 과정과 비슷합니다. 일반적으로 토론 문서 내 논의가 시작되고 프로젝트 문서 상단에 {{새로운 토론|문단=진행 중인 토론 문단}} 틀을 붙여 공동체의 참여를 요청합니다. 논의가 충분히 이루어진 후, 제3의 편집자가 토론을 종료하고 평가한 후 상태 변경 총의가 형성되었는지 판단해야 합니다. 폐지된 정책이나 지침은 최상단에 {{중단}} 틀을 붙여 더 이상 사용하지 않는 정책/지침임을 알립니다.
소수의 공동체 인원만 지지하는 수필, 정보문 및 기타 비공식 문서는 일반적으로 주된 작성자의 사용자 이름공간으로 이동합니다. 이러한 논의는 일반적으로 해당 문서의 토론란에서 이루어지며, 간혹 위키백과:의견 요청을 통해 처리되기도 합니다.
같이 보기
위키백과:위키백과의 정책과 지침 목록
위키백과:의견 요청
수필
위키백과:제품, 절차, 정책
위키백과:위키백과 공동체의 기대와 규범
기타 링크
위키백과:사랑방 (정책) - 현재는 폐지됨.
외부 링크
위키미디어 재단의 사명
위키미디어 재단이 추구하는 가치
meta:Founding principles - 메타위키의 창립원리
vte위키백과의 주요 정책과 지침 (?)
다섯 원칙
규칙에 얽매이지 마세요
컨텐츠 (?)P
확인 가능
독자 연구 금지
중립적 시각
위키백과에 대한 오해
생존 인물의 전기

In [9]:
print(splits[14].page_content)

정책 및 지침을 과감히 편집할 시 되돌리기를 당했다면 먼저 되돌리기 하는 것 보다, 토론을 열어 의견을 나누는 것이 좋습니다. 토론 중 자신의 주장을 뒷받침하기 위한 정책 편집은 체제를 시험하는 것으로 보일 수 있습니다. 특히 편집 시 논쟁에 자신이 참여한 사실을 밝히지 않았다면 더더욱이요.
격하
특정 정책이나 지침이 편집 관행이나 공동체 규범이 바뀌며 쓸모없어질 수 있고, 다른 문서가 개선되어 내용이 중복될 수 있으며, 불필요한 내용이 증식할 수도 있습니다. 이 경우 편집자들은 정책을 지침으로 격하하거나, 정책 또는 지침을 보충 설명, 정보문, 수필 또는 중단 문서로 격하할 것을 제안할 수 있습니다. 
격하 과정은 채택 과정과 비슷합니다. 일반적으로 토론 문서 내 논의가 시작되고 프로젝트 문서 상단에 {{새로운 토론|문단=진행 중인 토론 문단}} 틀을 붙여 공동체의 참여를 요청합니다. 논의가 충분히 이루어진 후, 제3의 편집자가 토론을 종료하고 평가한 후 상태 변경 총의가 형성되었는지 판단해야 합니다. 폐지된 정책이나 지침은 최상단에 {{중단}} 틀을 붙여 더 이상 사용하지 않는 정책/지침임을 알립니다.
소수의 공동체 인원만 지지하는 수필, 정보문 및 기타 비공식 문서는 일반적으로 주된 작성자의 사용자 이름공간으로 이동합니다. 이러한 논의는 일반적으로 해당 문서의 토론란에서 이루어지며, 간혹 위키백과:의견 요청을 통해 처리되기도 합니다.
같이 보기
위키백과:위키백과의 정책과 지침 목록
위키백과:의견 요청
수필
위키백과:제품, 절차, 정책
위키백과:위키백과 공동체의 기대와 규범
기타 링크
위키백과:사랑방 (정책) - 현재는 폐지됨.
외부 링크
위키미디어 재단의 사명
위키미디어 재단이 추구하는 가치
meta:Founding principles - 메타위키의 창립원리
vte위키백과의 주요 정책과 지침 (?)
다섯 원칙
규칙에 얽매이지 마세요
컨텐츠 (?)P
확인 가능
독자 연구 금지
중립적 시각
위키백과에 대한 오해
생존 인물의 전기
저작권
G
문서 등재 기준


In [10]:
print(splits[14].metadata)

{'source': 'https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EC%A0%95%EC%B1%85%EA%B3%BC_%EC%A7%80%EC%B9%A8', 'title': '위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전', 'language': 'ko'}


## 분할된 각각의 청크들을 벡터 스토어에 저장 및 인덱싱(Indexing)

Indexing: chunk => embedding => store

OpenAI의 텍스트 임베딩 모델을 사용할 수 있도록 OpenAIEmbeddings를 import 한다.  
자연어로 된 텍스트를 입력받아 의미적 유사성을 계산할 수 있는 숫자 배열로 변환한다. OPENAI_API_KEY 설정이 필요하다.

텍스트를 고차원 숫자 배열(벡터)로 변환해주는 임베딩 모델 객체를 생성하고 각 문서 청의 텍스트가 의미하는 바를 숫자로 매핑한다.  
임베딩 모델을 지정하지 않으면 `text-embedding-ada-002`가 기본 모델로 사용된다.

In [11]:
from langchain_openai import OpenAIEmbeddings

로컬 및 메모리 기반으로 동작하는 고성능 오픈소스 벡터 데이터베이스인 Chroma를 사용할 수 있도록 import 한다.  
OpenAIEmbeddings로 변환된 벡터 데이터를 메타 데이터와 함께 저장하고 사용자가 질문을 던졌을 때 유사도 검색을 수행하여 질문과 가장 의미가 가까운 청크를 찾아낸다.

In [12]:
from langchain_community.vectorstores import Chroma

문서 청크들(splits)을 OpenAI의 임베딩 모델을 사용해서 고차원 숫자 벡터로 변환한뒤, Chroma 벡터 데이터베이스에 저장한다.  
`documents`: RecursiveCharacterTextSplitter로 잘라둔 Document 객체 리스트(청크)를 지정한다.  
`embedding`: 텍스트를 벡터화 할 임베딩 모델을 지정한다.

[splits(문서 청크들)]  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓  
[OpenAIEmbeddings] => (OpenAI API 호출) => 텍스트가 벡터(숫자 배열)로 변환  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓  
[Chroma DB] => 메모리 인덱스에 (벡터 + 원본 텍스트 + 메타 데이터) 저장  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓  
vectorstores

In [13]:
vectorstores = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

## 검색 및 생성

Chroma 벡터 데이터베이스에서 similarity_search() 메소드를 실행하면 인수로 지정된 질문과 의미가 가장 유사한 문서 검색해서 가져온다.  
`k`: 기본값은 4, Chroma 벡터 데이터베이스에서 가져올 의미가 가장 유사한 문서의 개수를 지정한다.

In [14]:
docs = vectorstores.similarity_search('격하 과정에 대해서 설명해주세요.')

print(len(docs))
print(docs[0].page_content)

4
아래에서 보듯, 당신은 과감하게 삽입하거나, 토론을 통하여 광범위한 총의를 모으는 방식으로 모범적인 활동 규범을 제시할 수 있습니다.
실질적인 변경
실행하세요. 정책 및 지침 문서를 실질적으로 변경하기 전에, 기존 관행에 대한 합리적인 예외를 설정하는 것이 유용할 수 있습니다. 이러한 방식으로 기존의 관행을 갱신하기 위해, 백:과감과 백:무시 정신에 따라 기존의 관행에서 직접적으로 벗어날 수도 있습니다. 시간이 지나고, 변경에 대한 이의가 없거나 토론을 통해 변경 또는 구현에 대한 광범위한 총의에 도달한 경우, 관행을 설명하는 정책 및 지침 문서를 편집하여 새로운 상황을 반영할 수 있습니다.
토론 먼저 하세요. 토론 문서 내 논의는 일반적으로 정책의 실질적인 변경보다 우선합니다. 이의가 없거나 토론에서 해당 변경에 대한 총의가 형성되었다면 변경할 수 있습니다. 서식, 문법 및 명료성 개선을 위한 사소한 편집은 언제든지 가능합니다.
논의 결과가 불분명할 경우 제안 과정에서처럼 관리자나 다른 독립 편집자가 평가해야 합니다. 주요한 변화들은 또한 일반적으로 지역사회에 공표되어야 합니다; 제안 과정과 유사한 발표가 적절할 수 있습니다.
더욱 폭넓은 참여가 필요하다면 {{새로운 토론|문단=진행 중인 토론 문단}} 틀이 유용할 수 있습니다. 
또는 과감해지세요. 비록 편집자들 대부분이 (특히 잘 짜인 문서에서) 기존의 논의 기록을 찾지만, 이러한 문서를 직접 편집하는 것은 위키피디아의 정책에 의해 허용됩니다. 따라서 변경 전에 총의를 나타내는 정식 논의가 없었다는 이유만으로 변경 사항을 제거해서는 안 됩니다. 대신, 편집 요약이나 토론 문서에 실질적인 반박 이유를 제시해야 합니다.
정책 및 지침을 과감히 편집할 시 되돌리기를 당했다면 먼저 되돌리기 하는 것 보다, 토론을 열어 의견을 나누는 것이 좋습니다. 토론 중 자신의 주장을 뒷받침하기 위한 정책 편집은 체제를 시험하는 것으로 보일 수 있습니다. 특히 편집 시 논쟁에 자신이 참여한 사실을 밝히지 않았다면 더더욱이요

In [15]:
for doc in docs:
    print(doc)
    print('-' * 100)

page_content='아래에서 보듯, 당신은 과감하게 삽입하거나, 토론을 통하여 광범위한 총의를 모으는 방식으로 모범적인 활동 규범을 제시할 수 있습니다.
실질적인 변경
실행하세요. 정책 및 지침 문서를 실질적으로 변경하기 전에, 기존 관행에 대한 합리적인 예외를 설정하는 것이 유용할 수 있습니다. 이러한 방식으로 기존의 관행을 갱신하기 위해, 백:과감과 백:무시 정신에 따라 기존의 관행에서 직접적으로 벗어날 수도 있습니다. 시간이 지나고, 변경에 대한 이의가 없거나 토론을 통해 변경 또는 구현에 대한 광범위한 총의에 도달한 경우, 관행을 설명하는 정책 및 지침 문서를 편집하여 새로운 상황을 반영할 수 있습니다.
토론 먼저 하세요. 토론 문서 내 논의는 일반적으로 정책의 실질적인 변경보다 우선합니다. 이의가 없거나 토론에서 해당 변경에 대한 총의가 형성되었다면 변경할 수 있습니다. 서식, 문법 및 명료성 개선을 위한 사소한 편집은 언제든지 가능합니다.
논의 결과가 불분명할 경우 제안 과정에서처럼 관리자나 다른 독립 편집자가 평가해야 합니다. 주요한 변화들은 또한 일반적으로 지역사회에 공표되어야 합니다; 제안 과정과 유사한 발표가 적절할 수 있습니다.
더욱 폭넓은 참여가 필요하다면 {{새로운 토론|문단=진행 중인 토론 문단}} 틀이 유용할 수 있습니다. 
또는 과감해지세요. 비록 편집자들 대부분이 (특히 잘 짜인 문서에서) 기존의 논의 기록을 찾지만, 이러한 문서를 직접 편집하는 것은 위키피디아의 정책에 의해 허용됩니다. 따라서 변경 전에 총의를 나타내는 정식 논의가 없었다는 이유만으로 변경 사항을 제거해서는 안 됩니다. 대신, 편집 요약이나 토론 문서에 실질적인 반박 이유를 제시해야 합니다.
정책 및 지침을 과감히 편집할 시 되돌리기를 당했다면 먼저 되돌리기 하는 것 보다, 토론을 열어 의견을 나누는 것이 좋습니다. 토론 중 자신의 주장을 뒷받침하기 위한 정책 편집은 체제를 시험하는 것으로 보일 수 있습니다. 특히 편집 시 논쟁에 자신이 참여한 사실을 밝히